# Create and Edit Images with GPT-Image-2 on Azure AI Foundry

Content:
1. Create images using a text prompt
2. Create images with a transparent background
3. Edit an input image using a text prompt
4. Compose a new image based on several input images
5. Improve Input Fidelity
6. Mask-based inpainting

Resources:
- https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/dall-e
- https://developers.openai.com/api/docs/models/gpt-image-2
- https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/responses?tabs=python-secure#image-generation

## Setup
Use the project `.venv` created by `uv sync` as the notebook kernel. Authenticate first with `az login`.

In [ ]:
import requests
import os
from dotenv import load_dotenv, find_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

from utils import display_images

In [ ]:
if not load_dotenv(find_dotenv()): raise IOError("Error: .env file could not be loaded!")

ai_foundry_endpoint = os.environ["AI_FOUNDRY_ENDPOINT"]
imagegen_deployment = os.getenv("IMAGEGEN_2_DEPLOYMENT", "gpt-image-2")
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

aoai_client = AzureOpenAI(
    azure_endpoint=ai_foundry_endpoint,
    api_version="2025-04-01-preview",
    azure_ad_token_provider=token_provider,
)

## Create images using a text prompt

In [ ]:
result = aoai_client.images.generate(
    model=imagegen_deployment,
    prompt="A photograph of a red fox in an autumn forest",
    background="auto", # auto, transparent, opaque
    n=2,
    quality="high",
    size="1536x1024", # or any valid GPT-Image-2 WIDTHxHEIGHT (16px multiples, up to 3840px)
    output_format="png", # png, jpg, webp (webp not supported in AOAI)

)

images_data = result.model_dump()["data"]
encoded_images = [img["b64_json"] for img in images_data]

display_images(encoded_images, width=600)

In [ ]:
prompt = """
A wide image taken with a phone of a glass whiteboard in a modern office. The whiteboard is positioned against an interior wall, with a faint reflection of the Paris cityscape visible in the  glass. 
A woman is writing on the board, wearing a T-shirt with the text "AI GBB" on the back. The handwriting looks natural and slightly messy. The photographer’s reflection is also visible.

The text reads:
(left)
Business Perspective
- Industry context
- Stakeholder alignment
- Agreed success criteria
- Clear roadmap

AI Enablers
- Reasoning models
- Agentic AI
- Fine-Tuning
- Multi Modality

(right)
Process
Execute → Feedback → Iterate

Outcome
- Customer Value
- AI ACR
"""

result = aoai_client.images.generate(
    model=imagegen_deployment,
    prompt=prompt,
    n=1,
    quality="high",
    size="1536x1024",
    output_format="png",
)
images_data = result.model_dump()["data"]
encoded_images = [img["b64_json"] for img in images_data]

display_images(encoded_images, width=800)

## Create images with a transparent background
Images with transparent backgrounds are useful because they allow seamless layering over different backgrounds without visible edges or unwanted color blocks. This makes them ideal for logos, icons, and UI elements where flexibility and clean integration are important. Additionally, PNG preserves image quality without compression artifacts.

In [ ]:
prompt = "A commercial quality photograph of a premium bottle of champagne labeled 'Visionary Lab', standing upright. The bottle features a sleek, elegant label with gold accents."

result = aoai_client.images.generate(
    model=imagegen_deployment,
    prompt=prompt,
    n=3,
    quality="high", # auto, high, medium, low
    background="transparent", 
    size="1024x1536", # flexible GPT-Image-2 dimensions are also supported
)

images_data = result.model_dump()["data"]
encoded_images = [img["b64_json"] for img in images_data]

display_images(encoded_images, width=600)

## Edit an image using a text prompt
Edits are currently not supported with the OpenAI Python SDK when using Azure OpenAI. Therefore, we are switching to the REST API.

In [ ]:
source_image = "images/sportive-car.png"
display_images(source_image, width=800)

Note: Edits are not supported in AOAI with the Python SDK. We will have to use the REST API.

In [ ]:
url = f"{ai_foundry_endpoint.rstrip('/')}/openai/deployments/{imagegen_deployment}/images/edits?api-version=2025-04-01-preview"

headers = {
    "Authorization": f"Bearer {credential.get_token('https://cognitiveservices.azure.com/.default').token}"
}

In [ ]:
files = {
    "image": open(source_image, "rb"),
}
data = {
    "prompt": "make the car color light metallic blue and the background light grey/white",
    "n": 1,
    "size": "1536x1024",
    "quality": "high",
}

# Send the request
response = requests.post(url, headers=headers, files=files, data=data)
response.raise_for_status()

images_data = response.json()["data"]
encoded_images = [img["b64_json"] for img in images_data]
display_images(encoded_images, width=800)

## Compose a new image based on several input images

In [ ]:
source_images = ["images/shirt.png", "images/trousers.png", "images/shoes.png", "images/briefcase.png"]
display_images(source_images, width=500)    

In [ ]:
prompt = "Generate a photorealistic catalog image of a man wearing the four items in the reference pictures (shirt, trousers, shoes, briefcase)."

files = [("image[]", open(image, "rb")) for image in source_images]

data = {
    "prompt": prompt,
    "n": 4,
    "size": "1024x1536",
    "quality": "high",
}

response = requests.post(url, headers=headers, files=files, data=data)
response.raise_for_status()

images_data = response.json()["data"]
encoded_images = [img["b64_json"] for img in images_data]
display_images(encoded_images, width=500)

In [ ]:
source_images = ["images/woman-shirt.png", "images/woman-jeans.png", "images/woman-shoes.png", "images/woman.png"]
display_images(source_images, width=500)    

In [ ]:
prompt = """
You are provided with four reference images:
1. A portrait photo of a woman.
2. Three catalog images showing a shirt, jeans, and shoes.

Task:
Generate a high-quality, photorealistic catalog image of the woman, dressed in the referenced shirt, jeans, and shoes. Ensure the final image looks natural, consistent, and suitable for fashion catalog presentation.
"""

files = [("image[]", open(image, "rb")) for image in source_images]

data = {
    "prompt": prompt,
    "n": 4,
    "size": "1024x1536",
    "quality": "high",
}

response = requests.post(url, headers=headers, files=files, data=data)
response.raise_for_status()

images_data = response.json()["data"]
encoded_images = [img["b64_json"] for img in images_data]
display_images(encoded_images, width=500)

## Improve Input Fidelity
Azure OpenAI’s Input Fidelity control allows more accurate preservation of original facial features and other details during edits (though some imperfections remain).

In [ ]:
source_images = ["images/tattoed-man.png", "images/man-summer-shirt.png"] # ensure that the image with the details you care most about is the first item in the list
display_images(source_images, width=500)   

### Default Input Fidelity

In [ ]:
prompt = "The man in the summer shirt. Keep the background of the original image."

files = [("image[]", open(image, "rb")) for image in source_images]

data = {
    "prompt": prompt,
    "n": 2,
    "size": "1536x1024",
    "quality": "high",
    "model": imagegen_deployment,
    "input_fidelity": "low",  # low (default), high
}

response = requests.post(url, headers=headers, files=files, data=data)
response.raise_for_status()

images_data = response.json()["data"]
encoded_images = [img["b64_json"] for img in images_data]
display_images(encoded_images, width=1200)

### High Input Fidelity

In [ ]:
prompt = "The man in the summer shirt. Keep the background of the original image."

files = [("image[]", open(image, "rb")) for image in source_images]


data = {
    "prompt": prompt,
    "n": 2,
    "size": "1536x1024",
    "quality": "high",
    "model": imagegen_deployment,
    "input_fidelity": "high",  # high or low
}

response = requests.post(url, headers=headers, files=files, data=data)
response.raise_for_status()

images_data = response.json()["data"]
encoded_images = [img["b64_json"] for img in images_data]
display_images(encoded_images, width=1200)

## Mask-based inpainting

In [ ]:
source_image = "images/no-smile.png"
mask_image = "images/mask.png"
display_images([source_image, mask_image], width=500)

In [ ]:
files = {
    "image": open(source_image, "rb"),
    "mask": open(mask_image, "rb"),
}
data = {
    "prompt": "colorful polo shirt",
    "n": 1,
    "size": "1024x1024",
    "quality": "high",
}

# Send the request
response = requests.post(url, headers=headers, files=files, data=data)
response.raise_for_status()

In [ ]:
images_data = response.json()["data"]
encoded_images = [img["b64_json"] for img in images_data]
display_images(encoded_images, width=500)

## Responses API

In [ ]:
client = aoai_client

In [ ]:
# Responses API image-tool examples require a compatible orchestrator LLM
# deployment. This notebook intentionally tests GPT-Image-2 directly through
# the Images API so it runs with the documented Visionary Lab deployment set.
